# Episode 7: Middleware

Middleware is **cross-cutting behavior** — logging, retries, history trimming, auditing — applied consistently to every run without changing the agent, the model, or the tools.

**In this episode you'll build:** custom middleware that intercepts LLM calls and tool executions.

## Why middleware?

Some behavior doesn't belong to any one agent or tool — it belongs to *every turn*:

- Log what the agent is doing
- Retry when the provider has a transient failure
- Trim history before it reaches the model
- Audit every tool call

Putting that logic in the prompt or in each tool would scatter it everywhere. Middleware centralizes it in one place and plugs into the `middleware=` slot.

## The four hooks

A middleware subclasses `BaseMiddleware` and implements only the hooks it needs:

| Hook | Wraps | Use for |
|---|---|---|
| `on_turn` | The whole agent turn | Total latency, turn-level policy |
| `on_llm_call` | Each LLM API call | Retry, logging, history trim, caching |
| `on_tool_execution` | Each tool invocation | Validate args, redact results, access control |
| `on_human_input` | Each human-input request | Audit, rewrite prompts, rate limit |

Each hook receives a `call_next` — call it to continue down the stack, and you can run code before and after.

## Built-in middleware

You don't have to write everything. `autogen.beta.middleware` ships ready-made middleware:

| Middleware | Purpose |
|---|---|
| `LoggingMiddleware()` | Logs turn start/end, each LLM call, each tool run |
| `RetryMiddleware(max_retries=N)` | Retries failed LLM calls |
| `HistoryLimiter(max_events=N)` | Caps event history by count |
| `TokenLimiter(max_tokens=N)` | Caps history by rough token estimate |
| `TelemetryMiddleware(...)` | OpenTelemetry spans (Episode 24) |

Built-ins are constructed and dropped straight into `middleware=[...]`.

## Setup

In [ ]:
from dotenv import load_dotenv

load_dotenv()

from collections.abc import Sequence

from autogen.beta import Agent, Context
from autogen.beta.config import OpenAIConfig
from autogen.beta.events import BaseEvent, ModelResponse, ToolCallEvent
from autogen.beta.middleware import BaseMiddleware, LLMCall, ToolExecution, RetryMiddleware

config = OpenAIConfig(model="gpt-4.1-mini", temperature=0)


## A custom middleware: intercept the LLM call

Subclass `BaseMiddleware` and implement `on_llm_call`. The pattern is always the same: do something, `await call_next(...)`, do something with the result, return it.

To register a custom middleware, pass the **class** (not an instance) — the agent creates a fresh instance per turn.

In [1]:
class CallCounter(BaseMiddleware):
    """Counts and logs each LLM call in a turn."""

    async def on_llm_call(
        self, call_next: LLMCall, events: Sequence[BaseEvent], context: Context
    ) -> ModelResponse:
        print(f"[CallCounter] LLM call with {len(events)} events")
        response = await call_next(events, context)
        print("[CallCounter] LLM responded")
        return response


agent = Agent(
    "assistant",
    prompt="Reply in one short sentence.",
    config=config,
    middleware=[CallCounter],
)

reply = await agent.ask("What is 2 plus 2?")
print("REPLY:", reply.body)


[CallCounter] LLM call with 1 events
[CallCounter] LLM responded
REPLY: 2 plus 2 is 4.


## A custom middleware: intercept tool calls

The `on_tool_execution` hook fires once per tool call. Here a built-in (`RetryMiddleware`) and a custom auditor run together — middleware compose freely in the list.

In [2]:
def get_stock(symbol: str) -> str:
    """Get the current price of a stock symbol."""
    return f"{symbol}: $187.40"


class ToolAuditor(BaseMiddleware):
    """Logs every tool the agent runs."""

    async def on_tool_execution(
        self, call_next: ToolExecution, event: ToolCallEvent, context: Context
    ):
        print(f"[audit] tool requested: {event.name}")
        result = await call_next(event, context)
        print(f"[audit] tool {event.name} done")
        return result


agent = Agent(
    "trader",
    prompt="Use get_stock to answer. One sentence.",
    config=config,
    tools=[get_stock],
    middleware=[RetryMiddleware(max_retries=2), ToolAuditor],
)

reply = await agent.ask("What is the price of AAPL?")
print("REPLY:", reply.body)


[audit] tool requested: get_stock
[audit] tool get_stock done
REPLY: The current price of AAPL is $187.40.


## Ordering matters

Middleware runs in registration order, like nested `with` blocks. Registering `[A, B, C]` enters `A → B → C` and unwinds `C → B → A`.

That order is a real decision. If you want retries logged separately, put logging *inside* retry. If history-trimming should operate on what's actually sent, put it *closest* to the LLM call. A typical production stack:

```python
middleware=[
    TelemetryMiddleware(...),   # outermost: sees retries as separate spans
    LoggingMiddleware(),        # logs each retry
    RetryMiddleware(max_retries=2),
    HistoryLimiter(max_events=200),  # closest to the LLM call
]
```

## Additional content

**Call-level middleware.** `agent.ask(...)` and `reply.ask(...)` accept `middleware=` for one turn only — handy for ad-hoc debugging without changing the agent.

**Per-turn state.** A middleware instance is created once per turn, so it can store state on `self` (a counter, a start time) and use it across its hooks within that turn.

**For history shaping**, simple count/token caps (`HistoryLimiter`, `TokenLimiter`) live in middleware — but richer strategies (sliding windows, summaries, working memory) belong to **assembly policies**, which is the next episode.

## What's next

Middleware wraps the turn. **Assembly policies** shape what actually goes *into* the prompt each turn — that's Episode 8.